# AstroCLIMB metadata-first hybrid: **Unified Multi-Modal Model**

Major upgrade from the original three independent CatBoost specialists.

**Key change**: A single early-fusion neural network is trained on **all** pairs,
regardless of modality (text-text / text-image / image-image). This lets the model
learn cross-modality interactions and share capacity instead of fragmenting the data.

Pipeline:
1. Exact Hugging Face metadata matching
2. Candidate-set graph consensus for ambiguous matches
3. Unified early-fusion MLP on SPECTER2 + SigLIP2 + DINO + handcrafted features
4. Metadata-first inference with the unified neural fallback + submission

Use a Kaggle GPU. Attach the full competition `train.csv`, `test.csv`,
`sample_submission.csv`, this repository (for `metadata_matching.py`), and
model folders when Internet is disabled. DINOv3 is gated; accept its license
and set `HF_TOKEN`, or the notebook falls back to DINOv2 automatically.


In [ ]:
# Kaggle setup. Uncomment when packages are missing.
%pip install -q "transformers>=4.51" datasets pyarrow easyocr


In [ ]:
%%writefile metadata_matching.py
"""Match AstroCLIMB Kaggle objects to Hugging Face metadata.

The matcher builds compact caption and image-hash indexes from an AstroCLIMB
Parquet file, then streams a Kaggle pair CSV in small chunks.  It never writes
the Base64 objects to its output.  Exact metadata matches are used to infer the
four competition relationships from figure identity, DOI equality, and the
citation graph.

Example:
  python metadata_matching.py \
      --pairs data/train_1000.csv \
      --hf-dataset adsabs/AstroCLIMB \
      --metadata-only \
      --output results/train_metadata_matches.csv
"""

from __future__ import annotations

import argparse
import base64
import hashlib
import io
import json
import os
import pickle
import re
import struct
import unicodedata
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Iterable

import pandas as pd
from PIL import Image, ImageOps


LABELS = (
  "same_figure",
  "same_paper",
  "related_papers",
  "unrelated_papers",
)


def normalize_caption(value: Any) -> str:
  """Normalize formatting without removing scientifically meaningful text."""
  if value is None:
      return ""
  text = unicodedata.normalize("NFKC", str(value))
  return re.sub(r"\s+", " ", text).strip().casefold()


def normalize_doi(value: Any) -> str:
  if value is None:
      return ""
  doi = str(value).strip().casefold()
  doi = re.sub(r"^https?://(?:dx\.)?doi\.org/", "", doi)
  return doi.rstrip(".,; ")


def doi_set(value: Any) -> frozenset[str]:
  """Convert Parquet list, numpy array, or JSON-like DOI value to a set."""
  if value is None:
      return frozenset()
  if isinstance(value, str):
      stripped = value.strip()
      if not stripped:
          return frozenset()
      if stripped.startswith("["):
          try:
              value = json.loads(stripped)
          except json.JSONDecodeError:
              value = [stripped]
      else:
          value = [stripped]
  try:
      values: Iterable[Any] = list(value)
  except TypeError:
      values = [value]
  return frozenset(filter(None, (normalize_doi(item) for item in values)))


def sha256_bytes(raw: bytes) -> str:
  return hashlib.sha256(raw).hexdigest()


def caption_key(value: Any) -> str:
  """Use a fixed-size key instead of retaining full captions in the index."""
  return sha256_bytes(normalize_caption(value).encode("utf-8"))


def pixel_sha256(raw: bytes) -> str:
  """Hash decoded pixels so equivalent PNG encodings can still match."""
  with Image.open(io.BytesIO(raw)) as image:
      image = ImageOps.exif_transpose(image).convert("RGB")
      size = struct.pack(">II", image.width, image.height)
      return sha256_bytes(size + image.tobytes())


def is_base64_image(value: Any) -> bool:
  if not isinstance(value, str):
      return False
  stripped = value.lstrip()
  return stripped.startswith("iVBOR") or stripped.startswith("data:image/")


def decode_base64_image(value: str) -> bytes:
  payload = value.strip()
  if payload.startswith("data:image/"):
      payload = payload.split(",", 1)[1]
  return base64.b64decode(payload, validate=False)


def parquet_image_bytes(value: Any, base_dir: Path) -> bytes | None:
  """Read common Hugging Face Image representations returned by Parquet."""
  if value is None:
      return None
  if isinstance(value, bytes):
      return value
  if isinstance(value, (bytearray, memoryview)):
      return bytes(value)
  if isinstance(value, Image.Image):
      buffer = io.BytesIO()
      value.save(buffer, format="PNG")
      return buffer.getvalue()
  if isinstance(value, dict):
      raw = value.get("bytes")
      if raw is not None:
          return bytes(raw)
      path = value.get("path")
      if path:
          image_path = Path(path)
          if not image_path.is_absolute():
              image_path = base_dir / image_path
          return image_path.read_bytes()
  return None


def collect_pair_image_hashes(
  pairs_path: Path,
  chunk_size: int,
  limit: int | None,
  include_pixel_hashes: bool,
) -> tuple[set[str], set[str]]:
  """Collect compact hashes for only the images that occur in the pair CSV."""
  raw_hashes: set[str] = set()
  pixel_hashes: set[str] = set()
  processed = 0
  for chunk in pd.read_csv(
      pairs_path,
      usecols=["obj_1", "obj_2"],
      chunksize=chunk_size,
      keep_default_na=False,
  ):
      if limit is not None:
          remaining = limit - processed
          if remaining <= 0:
              break
          chunk = chunk.iloc[:remaining]
      for column in ("obj_1", "obj_2"):
          for value in chunk[column]:
              if not is_base64_image(value):
                  continue
              try:
                  raw = decode_base64_image(value)
              except Exception:
                  continue
              raw_hashes.add(sha256_bytes(raw))
              if include_pixel_hashes:
                  try:
                      pixel_hashes.add(pixel_sha256(raw))
                  except Exception:
                      pass
      processed += len(chunk)
  print(
      f"Pair image targets: {len(raw_hashes)} raw hashes"
      + (f", {len(pixel_hashes)} pixel hashes" if include_pixel_hashes else "")
  )
  return raw_hashes, pixel_hashes


def append_index(index: dict[str, list[int]], key: str, row_id: int) -> None:
  if key and row_id not in index[key]:
      index[key].append(row_id)


@dataclass(frozen=True)
class Match:
  method: str
  candidates: tuple[int, ...]

  @property
  def unique_row(self) -> int | None:
      return self.candidates[0] if len(self.candidates) == 1 else None


class MetadataIndex:
  def __init__(
      self,
      metadata_path: Path,
      include_images: bool = True,
      include_pixel_hashes: bool = False,
      parquet_batch_size: int = 256,
  ) -> None:
      try:
          import pyarrow.dataset as pyarrow_dataset
      except ImportError as exc:
          raise SystemExit(
              "Reading Parquet requires pyarrow. Install it with: pip install pyarrow"
          ) from exc

      dataset = pyarrow_dataset.dataset(str(metadata_path), format="parquet")
      required = {"image", "Image Caption", "Paper DOI", "Image ID"}
      missing = required.difference(dataset.schema.names)
      if missing:
          raise ValueError(f"Metadata is missing columns: {sorted(missing)}")

      self.records: list[dict[str, Any]] = []
      self.caption_index: dict[str, list[int]] = defaultdict(list)
      self.raw_image_index: dict[str, list[int]] = defaultdict(list)
      self.pixel_image_index: dict[str, list[int]] = defaultdict(list)
      self.includes_images = include_images
      self.includes_pixel_hashes = include_images and include_pixel_hashes

      metadata_columns = [
          column
          for column in (
              "UUID",
              "Image ID",
              "Paper DOI",
              "Image Caption",
              "References DOIs",
              "Citing DOIs",
          )
          if column in dataset.schema.names
      ]
      scanner = dataset.scanner(
          columns=metadata_columns,
          batch_size=max(parquet_batch_size, 256),
          use_threads=False,
      )
      uuid_to_internal: dict[str, int] = {}
      for batch in scanner.to_batches():
          values = batch.to_pydict()
          for offset in range(batch.num_rows):
              def get(column: str, default: Any = "") -> Any:
                  return values[column][offset] if column in values else default

              caption = str(get("Image Caption") or "")
              record = {
                  "meta_row": len(self.records),
                  "uuid": str(get("UUID") or ""),
                  "image_id": str(get("Image ID") or ""),
                  "doi": normalize_doi(get("Paper DOI")),
                  "references": doi_set(get("References DOIs", [])),
                  "citations": doi_set(get("Citing DOIs", [])),
              }
              internal_id = len(self.records)
              self.records.append(record)
              if record["uuid"]:
                  uuid_to_internal[record["uuid"]] = internal_id
              append_index(
                  self.caption_index,
                  caption_key(caption),
                  internal_id,
              )
          if len(self.records) % 10_000 < batch.num_rows:
              print(f"Indexed {len(self.records)} metadata rows")

      if include_images:
          print(
              "Building the image hash index. This streams the image column once; "
              "for the full dataset it reads approximately 72 GB."
          )
          base_dir = metadata_path if metadata_path.is_dir() else metadata_path.parent
          image_columns = ["image"]
          if "UUID" in dataset.schema.names:
              image_columns.insert(0, "UUID")
          image_scanner = dataset.scanner(
              columns=image_columns,
              batch_size=min(max(parquet_batch_size, 1), 32),
              use_threads=False,
          )
          sequential_row = 0
          for batch in image_scanner.to_batches():
              values = batch.to_pydict()
              for offset in range(batch.num_rows):
                  uuid = str(values.get("UUID", [""] * batch.num_rows)[offset] or "")
                  internal_id = uuid_to_internal.get(uuid, sequential_row)
                  sequential_row += 1
                  raw = parquet_image_bytes(values["image"][offset], base_dir)
                  if not raw:
                      continue
                  append_index(self.raw_image_index, sha256_bytes(raw), internal_id)
                  if include_pixel_hashes:
                      try:
                          append_index(self.pixel_image_index, pixel_sha256(raw), internal_id)
                      except Exception as exc:
                          print(f"Warning: metadata image {internal_id} could not be decoded: {exc}")
              if sequential_row % 1_000 < batch.num_rows:
                  print(f"Hashed {sequential_row} metadata images")

      print(
          "Metadata index:",
          {
              "rows": len(self.records),
              "captions": len(self.caption_index),
              "raw_images": len(self.raw_image_index),
              "pixel_images": len(self.pixel_image_index),
          },
      )

  def lookup(self, value: Any) -> Match:
      if is_base64_image(value):
          try:
              raw = decode_base64_image(value)
          except Exception:
              return Match("invalid_base64_image", ())

          candidates = tuple(self.raw_image_index.get(sha256_bytes(raw), ()))
          if candidates:
              method = "image_raw_exact" if len(candidates) == 1 else "image_raw_ambiguous"
              return Match(method, candidates)

          if not self.pixel_image_index:
              return Match("image_unmatched", ())
          try:
              candidates = tuple(self.pixel_image_index.get(pixel_sha256(raw), ()))
          except Exception:
              return Match("invalid_decoded_image", ())
          method = "image_pixel_exact" if len(candidates) == 1 else (
              "image_pixel_ambiguous" if candidates else "image_unmatched"
          )
          return Match(method, candidates)

      candidates = tuple(self.caption_index.get(caption_key(value), ()))
      method = "caption_exact" if len(candidates) == 1 else (
          "caption_ambiguous" if candidates else "caption_unmatched"
      )
      return Match(method, candidates)

  def record(self, match: Match) -> dict[str, Any] | None:
      row_id = match.unique_row
      return self.records[row_id] if row_id is not None else None


class HuggingFaceMetadataIndex(MetadataIndex):
  """Stream a Hugging Face dataset without downloading it in full first."""

  def __init__(
      self,
      dataset_name: str,
      split: str = "train",
      include_images: bool = False,
      include_pixel_hashes: bool = False,
  ) -> None:
      try:
          from datasets import Image as HuggingFaceImage
          from datasets import load_dataset
      except ImportError as exc:
          raise SystemExit(
              "Hugging Face streaming requires datasets. Install it with: "
              "pip install datasets"
          ) from exc

      self.records: list[dict[str, Any]] = []
      self.caption_index: dict[str, list[int]] = defaultdict(list)
      self.raw_image_index: dict[str, list[int]] = defaultdict(list)
      self.pixel_image_index: dict[str, list[int]] = defaultdict(list)
      self.includes_images = include_images
      self.includes_pixel_hashes = include_images and include_pixel_hashes

      print(f"Streaming metadata from {dataset_name!r}, split {split!r}")
      stream = load_dataset(dataset_name, split=split, streaming=True)
      required = {"image", "Image Caption", "Paper DOI", "Image ID"}
      missing = required.difference(stream.column_names)
      if missing:
          raise ValueError(f"Hugging Face dataset is missing columns: {sorted(missing)}")

      metadata_columns = [
          column
          for column in (
              "UUID",
              "Image ID",
              "Paper DOI",
              "Image Caption",
              "References DOIs",
              "Citing DOIs",
          )
          if column in stream.column_names
      ]
      uuid_to_internal: dict[str, int] = {}
      for row in stream.select_columns(metadata_columns):
          caption = str(row.get("Image Caption") or "")
          record = {
              "meta_row": len(self.records),
              "uuid": str(row.get("UUID") or ""),
              "image_id": str(row.get("Image ID") or ""),
              "doi": normalize_doi(row.get("Paper DOI")),
              "references": doi_set(row.get("References DOIs", [])),
              "citations": doi_set(row.get("Citing DOIs", [])),
          }
          internal_id = len(self.records)
          self.records.append(record)
          if record["uuid"]:
              uuid_to_internal[record["uuid"]] = internal_id
          append_index(self.caption_index, caption_key(caption), internal_id)
          if len(self.records) % 10_000 == 0:
              print(f"Indexed {len(self.records)} Hugging Face metadata rows")

      if include_images:
          print(
              "Streaming the full Hugging Face image column once. "
              "This transfers approximately 72 GB."
          )
          image_stream = load_dataset(dataset_name, split=split, streaming=True)
          image_stream = image_stream.cast_column(
              "image", HuggingFaceImage(decode=False)
          )
          image_columns = ["image"]
          if "UUID" in image_stream.column_names:
              image_columns.insert(0, "UUID")
          sequential_row = 0
          for row in image_stream.select_columns(image_columns):
              uuid = str(row.get("UUID") or "")
              internal_id = uuid_to_internal.get(uuid, sequential_row)
              sequential_row += 1
              raw = parquet_image_bytes(row.get("image"), Path("."))
              if not raw:
                  continue
              append_index(self.raw_image_index, sha256_bytes(raw), internal_id)
              if include_pixel_hashes:
                  try:
                      append_index(
                          self.pixel_image_index,
                          pixel_sha256(raw),
                          internal_id,
                      )
                  except Exception as exc:
                      print(
                          f"Warning: Hugging Face image {internal_id} "
                          f"could not be decoded: {exc}"
                      )
              if sequential_row % 1_000 == 0:
                  print(f"Hashed {sequential_row} Hugging Face images")

      print(
          "Metadata index:",
          {
              "rows": len(self.records),
              "captions": len(self.caption_index),
              "raw_images": len(self.raw_image_index),
              "pixel_images": len(self.pixel_image_index),
          },
      )


def load_or_build_metadata_index(
  metadata_path: Path | None,
  hf_dataset: str | None,
  hf_split: str,
  cache_path: Path | None,
  include_images: bool,
  include_pixel_hashes: bool,
  parquet_batch_size: int,
  rebuild: bool,
) -> MetadataIndex:
  if cache_path is not None and cache_path.exists() and not rebuild:
      print(f"Loading cached metadata index from {cache_path}")
      with cache_path.open("rb") as handle:
          cached = pickle.load(handle)
      cache_is_sufficient = not include_images or (
          getattr(cached, "includes_images", False)
          and (
              not include_pixel_hashes
              or getattr(cached, "includes_pixel_hashes", False)
          )
      )
      if cache_is_sufficient:
          return cached
      print("Cached index lacks requested image hashes; rebuilding it")

  if metadata_path is not None:
      index = MetadataIndex(
          metadata_path,
          include_images=include_images,
          include_pixel_hashes=include_pixel_hashes,
          parquet_batch_size=parquet_batch_size,
      )
  elif hf_dataset:
      index = HuggingFaceMetadataIndex(
          dataset_name=hf_dataset,
          split=hf_split,
          include_images=include_images,
          include_pixel_hashes=include_pixel_hashes,
      )
  else:
      raise ValueError("Provide either --metadata or --hf-dataset")
  if cache_path is not None:
      cache_path.parent.mkdir(parents=True, exist_ok=True)
      temporary = cache_path.with_suffix(cache_path.suffix + ".tmp")
      with temporary.open("wb") as handle:
          pickle.dump(index, handle, protocol=pickle.HIGHEST_PROTOCOL)
      os.replace(temporary, cache_path)
      print(f"Saved reusable metadata index to {cache_path}")
  return index


def add_target_image_hashes(
  index: MetadataIndex,
  metadata_path: Path | None,
  hf_dataset: str | None,
  hf_split: str,
  target_raw_hashes: set[str],
  target_pixel_hashes: set[str],
  image_batch_size: int,
) -> None:
  """Stream metadata images in batches, retaining only requested pair hashes."""
  completed_raw = set(getattr(index, "targeted_raw_hashes_scanned", set()))
  completed_pixels = set(getattr(index, "targeted_pixel_hashes_scanned", set()))
  requested_raw = target_raw_hashes.difference(completed_raw)
  requested_pixels = target_pixel_hashes.difference(completed_pixels)
  missing_raw = set(requested_raw)
  missing_pixels = set(requested_pixels)
  if not requested_raw and not requested_pixels:
      print("All requested pair-image hashes are already cached")
      return

  uuid_to_internal = {
      record["uuid"]: row_id
      for row_id, record in enumerate(index.records)
      if record["uuid"]
  }
  scanned = 0
  matched_raw = 0
  matched_pixels = 0

  def consume(values: dict[str, list[Any]], base_dir: Path) -> bool:
      nonlocal scanned, matched_raw, matched_pixels
      images = values.get("image", [])
      uuids = values.get("UUID", [""] * len(images))
      for offset, value in enumerate(images):
          internal_id = uuid_to_internal.get(str(uuids[offset] or ""), scanned)
          scanned += 1
          raw = parquet_image_bytes(value, base_dir)
          if not raw:
              continue
          raw_hash = sha256_bytes(raw)
          if raw_hash in requested_raw:
              append_index(index.raw_image_index, raw_hash, internal_id)
              if raw_hash in missing_raw:
                  missing_raw.discard(raw_hash)
                  matched_raw += 1
          if requested_pixels:
              try:
                  pixel_hash = pixel_sha256(raw)
              except Exception:
                  pixel_hash = ""
              if pixel_hash in requested_pixels:
                  append_index(index.pixel_image_index, pixel_hash, internal_id)
                  if pixel_hash in missing_pixels:
                      missing_pixels.discard(pixel_hash)
                      matched_pixels += 1
      if scanned % 1_000 < len(images):
          print(
              f"Scanned {scanned} metadata images; matched "
              f"{matched_raw} raw and {matched_pixels} pixel targets"
          )
      # Scan the entire metadata split so duplicate hashes remain ambiguous
      # instead of being incorrectly reported as unique matches.
      return False

  batch_size = max(1, image_batch_size)
  if metadata_path is not None:
      try:
          import pyarrow.dataset as pyarrow_dataset
      except ImportError as exc:
          raise SystemExit(
              "Reading Parquet requires pyarrow. Install it with: pip install pyarrow"
          ) from exc
      dataset = pyarrow_dataset.dataset(str(metadata_path), format="parquet")
      columns = ["image"]
      if "UUID" in dataset.schema.names:
          columns.insert(0, "UUID")
      scanner = dataset.scanner(columns=columns, batch_size=batch_size, use_threads=False)
      base_dir = metadata_path if metadata_path.is_dir() else metadata_path.parent
      for batch in scanner.to_batches():
          if consume(batch.to_pydict(), base_dir):
              break
  elif hf_dataset:
      try:
          from datasets import Image as HuggingFaceImage
          from datasets import load_dataset
      except ImportError as exc:
          raise SystemExit(
              "Hugging Face streaming requires datasets. Install it with: "
              "pip install datasets"
          ) from exc
      print(
          f"Batch-matching pair images from {hf_dataset!r}, split {hf_split!r}; "
          "only matching hashes will be retained"
      )
      stream = load_dataset(hf_dataset, split=hf_split, streaming=True)
      stream = stream.cast_column("image", HuggingFaceImage(decode=False))
      columns = ["image"]
      if "UUID" in stream.column_names:
          columns.insert(0, "UUID")
      for batch in stream.select_columns(columns).iter(batch_size=batch_size):
          if consume(batch, Path(".")):
              break
  else:
      raise ValueError("Provide either --metadata or --hf-dataset")

  index.targeted_raw_hashes_scanned = completed_raw.union(requested_raw)
  index.targeted_pixel_hashes_scanned = completed_pixels.union(requested_pixels)

  print(
      "Targeted image pass:",
      {
          "metadata_images_scanned": scanned,
          "raw_targets_matched": matched_raw,
          "pixel_targets_matched": matched_pixels,
          "raw_targets_unmatched": len(missing_raw),
          "pixel_targets_unmatched": len(missing_pixels),
      },
  )


def infer_relationship(left: dict[str, Any] | None, right: dict[str, Any] | None) -> str:
  if left is None or right is None:
      return "unmatched"
  if left["meta_row"] == right["meta_row"]:
      return "same_figure"
  left_doi, right_doi = left["doi"], right["doi"]
  if left_doi and left_doi == right_doi:
      return "same_paper"
  if left_doi and right_doi and (
      right_doi in left["references"]
      or right_doi in left["citations"]
      or left_doi in right["references"]
      or left_doi in right["citations"]
  ):
      return "related_papers"
  return "unrelated_papers"


def true_relationship(row: pd.Series) -> str:
  active = [label for label in LABELS if str(row.get(label, "0")) in {"1", "1.0"}]
  return active[0] if len(active) == 1 else "unknown"


def match_output_fields(prefix: str, match: Match, record: dict[str, Any] | None) -> dict[str, Any]:
  return {
      f"{prefix}_match_method": match.method,
      f"{prefix}_candidate_count": len(match.candidates),
      f"{prefix}_meta_row": "" if record is None else record["meta_row"],
      f"{prefix}_uuid": "" if record is None else record["uuid"],
      f"{prefix}_image_id": "" if record is None else record["image_id"],
      f"{prefix}_doi": "" if record is None else record["doi"],
  }


def bootstrap_same_figure_image(
  index: MetadataIndex,
  image_value: Any,
  image_match: Match,
  caption_record: dict[str, Any] | None,
) -> Match:
  """Learn an image-to-record mapping from a labeled same-figure train pair."""
  if image_match.unique_row is not None or caption_record is None:
      return image_match
  if not is_base64_image(image_value):
      return image_match
  try:
      raw_hash = sha256_bytes(decode_base64_image(image_value))
  except Exception:
      return image_match
  append_index(index.raw_image_index, raw_hash, caption_record["meta_row"])
  return index.lookup(image_value)


def match_pairs(
  metadata_path: Path | None,
  hf_dataset: str | None,
  hf_split: str,
  pairs_path: Path,
  output_path: Path,
  chunk_size: int,
  limit: int | None,
  index_cache: Path | None,
  include_images: bool,
  include_pixel_hashes: bool,
  targeted_image_matching: bool,
  image_batch_size: int,
  parquet_batch_size: int,
  rebuild_index: bool,
) -> None:
  index = load_or_build_metadata_index(
      metadata_path=metadata_path,
      hf_dataset=hf_dataset,
      hf_split=hf_split,
      cache_path=index_cache,
      include_images=include_images,
      include_pixel_hashes=include_pixel_hashes,
      parquet_batch_size=parquet_batch_size,
      rebuild=rebuild_index,
  )
  header = pd.read_csv(pairs_path, nrows=0).columns.tolist()
  required = {"id", "obj_1", "obj_2"}
  missing = required.difference(header)
  if missing:
      raise ValueError(f"Pair CSV is missing columns: {sorted(missing)}")

  if targeted_image_matching:
      target_raw_hashes, target_pixel_hashes = collect_pair_image_hashes(
          pairs_path=pairs_path,
          chunk_size=chunk_size,
          limit=limit,
          include_pixel_hashes=include_pixel_hashes,
      )
      add_target_image_hashes(
          index=index,
          metadata_path=metadata_path,
          hf_dataset=hf_dataset,
          hf_split=hf_split,
          target_raw_hashes=target_raw_hashes,
          target_pixel_hashes=target_pixel_hashes,
          image_batch_size=image_batch_size,
      )

  usecols = ["id", "obj_1", "obj_2"] + [label for label in LABELS if label in header]
  output_path.parent.mkdir(parents=True, exist_ok=True)
  temporary = output_path.with_suffix(output_path.suffix + ".tmp")
  if temporary.exists():
      temporary.unlink()

  processed = 0
  first_output = True
  for chunk in pd.read_csv(
      pairs_path,
      usecols=usecols,
      chunksize=chunk_size,
      keep_default_na=False,
  ):
      if limit is not None:
          remaining = limit - processed
          if remaining <= 0:
              break
          chunk = chunk.iloc[:remaining]

      output_rows = []
      for _, row in chunk.iterrows():
          left_match = index.lookup(row["obj_1"])
          right_match = index.lookup(row["obj_2"])
          left_record = index.record(left_match)
          right_record = index.record(right_match)
          truth = true_relationship(row)
          if truth == "same_figure":
              if is_base64_image(row["obj_1"]) and not is_base64_image(row["obj_2"]):
                  left_match = bootstrap_same_figure_image(
                      index, row["obj_1"], left_match, right_record
                  )
                  left_record = index.record(left_match)
              elif is_base64_image(row["obj_2"]) and not is_base64_image(row["obj_1"]):
                  right_match = bootstrap_same_figure_image(
                      index, row["obj_2"], right_match, left_record
                  )
                  right_record = index.record(right_match)
          inferred = infer_relationship(left_record, right_record)
          result = {
              "id": row["id"],
              "obj_1_modality": "image" if is_base64_image(row["obj_1"]) else "text",
              "obj_2_modality": "image" if is_base64_image(row["obj_2"]) else "text",
              "inferred_relationship": inferred,
              "true_relationship": truth,
              "metadata_correct": "" if truth == "unknown" or inferred == "unmatched" else inferred == truth,
          }
          result.update(match_output_fields("obj_1", left_match, left_record))
          result.update(match_output_fields("obj_2", right_match, right_record))
          output_rows.append(result)

      pd.DataFrame(output_rows).to_csv(
          temporary,
          mode="w" if first_output else "a",
          header=first_output,
          index=False,
      )
      first_output = False
      processed += len(chunk)
      print(f"Processed {processed} pairs")

  os.replace(temporary, output_path)
  print(f"Saved compact matches to {output_path}")

  if index_cache is not None:
      index_cache.parent.mkdir(parents=True, exist_ok=True)
      cache_temporary = index_cache.with_suffix(index_cache.suffix + ".tmp")
      with cache_temporary.open("wb") as handle:
          pickle.dump(index, handle, protocol=pickle.HIGHEST_PROTOCOL)
      os.replace(cache_temporary, index_cache)
      print(f"Updated metadata index cache at {index_cache}")

  audit = pd.read_csv(output_path, keep_default_na=False)
  both_matched = (
      audit["obj_1_candidate_count"].eq(1)
      & audit["obj_2_candidate_count"].eq(1)
  )
  inferred = audit["inferred_relationship"].ne("unmatched")
  known_truth = audit["true_relationship"].ne("unknown")
  summary = {
      "pairs": len(audit),
      "both_objects_uniquely_matched": int(both_matched.sum()),
      "both_objects_match_rate": round(float(both_matched.mean()), 4),
      "relationship_inference_coverage": round(float(inferred.mean()), 4),
  }
  scored = inferred & known_truth
  if scored.any():
      summary["accuracy_when_inferred"] = round(
          float((audit.loc[scored, "inferred_relationship"] == audit.loc[scored, "true_relationship"]).mean()),
          4,
      )
  print("Summary:", summary)
  methods = Counter(audit["obj_1_match_method"]) + Counter(audit["obj_2_match_method"])
  print("Object match methods:", dict(methods))


def parse_args() -> argparse.Namespace:
  parser = argparse.ArgumentParser(description=__doc__)
  parser.add_argument(
      "--metadata",
      type=Path,
      default=None,
      help="Optional local Parquet file/directory; overrides --hf-dataset",
  )
  parser.add_argument(
      "--hf-dataset",
      default="adsabs/AstroCLIMB",
      help="Hugging Face dataset streamed when --metadata is not provided",
  )
  parser.add_argument("--hf-split", default="train")
  parser.add_argument(
      "--pairs",
      type=Path,
      default=None,
      help="Explicit pair CSV path; otherwise locate --competition-file under /kaggle/input",
  )
  parser.add_argument(
      "--competition-file",
      choices=("train.csv", "test.csv"),
      default="train.csv",
  )
  parser.add_argument(
      "--output",
      type=Path,
      default=None,
  )
  parser.add_argument("--chunk-size", type=int, default=32)
  parser.add_argument("--parquet-batch-size", type=int, default=256)
  parser.add_argument(
      "--image-batch-size",
      type=int,
      default=32,
      help="Number of Hugging Face/Parquet images handled at a time",
  )
  parser.add_argument("--limit", type=int, default=None, help="Optional smoke-test row limit")
  parser.add_argument(
      "--index-cache",
      type=Path,
      default=None,
      help="Persistent metadata/hash index; reused by later train and test runs",
  )
  parser.add_argument(
      "--metadata-only",
      action="store_true",
      help="Skip the default targeted, batch-by-batch image matching pass",
  )
  parser.add_argument(
      "--full-image-index",
      action="store_true",
      help="Retain hashes for every metadata image instead of only pair-image targets",
  )
  parser.add_argument(
      "--pixel-hashes",
      action="store_true",
      help="Also decode all 94k metadata images for format-independent pixel hashes",
  )
  parser.add_argument(
      "--rebuild-index",
      action="store_true",
      help="Ignore and replace an existing index cache",
  )
  return parser.parse_args()


def find_competition_file(filename: str) -> Path:
  search_roots = [Path("/kaggle/input"), Path("data")]
  matches: list[Path] = []
  for root in search_roots:
      if root.exists():
          matches.extend(path for path in root.rglob(filename) if path.is_file())
  if not matches:
      raise FileNotFoundError(
          f"Could not find {filename!r}. Attach the AstroCLIMB competition "
          "data to the Kaggle notebook or pass --pairs explicitly."
      )
  matches.sort(key=lambda path: ("competitions" not in path.parts, len(path.parts), str(path)))
  print(f"Using competition file: {matches[0]}")
  if len(matches) > 1:
      print("Other matches:", [str(path) for path in matches[1:]])
  return matches[0]


if __name__ == "__main__":
  arguments = parse_args()
  pairs_path = arguments.pairs or find_competition_file(arguments.competition_file)
  working = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("results")
  output_path = arguments.output or working / f"{pairs_path.stem}_metadata_matches.csv"
  index_cache = arguments.index_cache or working / "astroclimb_metadata_index.pkl"
  include_images = arguments.full_image_index and not arguments.metadata_only
  targeted_image_matching = not arguments.metadata_only and not include_images
  match_pairs(
      metadata_path=arguments.metadata,
      hf_dataset=arguments.hf_dataset,
      hf_split=arguments.hf_split,
      pairs_path=pairs_path,
      output_path=output_path,
      chunk_size=arguments.chunk_size,
      limit=arguments.limit,
      index_cache=index_cache,
      include_images=include_images,
      include_pixel_hashes=arguments.pixel_hashes,
      targeted_image_matching=targeted_image_matching,
      image_batch_size=arguments.image_batch_size,
      parquet_batch_size=arguments.parquet_batch_size,
      rebuild_index=arguments.rebuild_index,
  )


In [ ]:
import base64, gc, hashlib, importlib.util, io, json, os, pickle, sqlite3, subprocess, sys, warnings
from collections import Counter
from difflib import SequenceMatcher
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageFile
from scipy.fft import dctn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_class_weight
from transformers import AutoImageProcessor, AutoModel, AutoProcessor, AutoTokenizer

ImageFile.LOAD_TRUNCATED_IMAGES = True
warnings.filterwarnings("ignore", message=".*DecompressionBomb.*")

LABELS = ["same_figure", "same_paper", "related_papers", "unrelated_papers"]
SEED = 2026
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
CSV_CHUNK = 4
MODEL_BATCH = 8
N_FOLDS = 5
USE_OCR = True
RUN_METADATA_MATCHER = True
RUN_CANDIDATE_CONSENSUS = True
REFRESH_METADATA_MATCHES = True  # never trust archived train/test match CSVs
REBUILD_METADATA_INDEX = True    # refresh the HF snapshot/index on the first match
MIN_TEST_METADATA_COVERAGE = 0.90

HF_DATASET = "adsabs/AstroCLIMB"
METADATA_PATH = None  # e.g. Path("/kaggle/input/current-astroclimb/AstroCLIMB.parquet")
SPECTER_MODEL = "allenai/specter2_base"
SIGLIP_MODEL = "google/siglip2-base-patch16-naflex"
DINO3_MODEL = "facebook/dinov3-vits16-pretrain-lvd1689m"
DINO_FALLBACK = "facebook/dinov2-small"

WORK = Path("/kaggle/working/astroclimb_hybrid") if Path("/kaggle/working").exists() else Path("results/astroclimb_hybrid")
WORK.mkdir(parents=True, exist_ok=True)
# The archive is only an embedding/feature cache. Its metadata matches and
# metadata index may refer to an older HF snapshot, so do not copy those.
SOURCE_DIR = Path("/kaggle/input/datasets/syedmohaiminulhoque/ressss/astroclimb_hybrid")
SAFE_CACHE_FILES = {"representations.sqlite", "tfidf.joblib", "train_features.npz", "test_features.npz"}
if SOURCE_DIR.exists():
    import shutil
    for name in SAFE_CACHE_FILES:
        src, dst = SOURCE_DIR / name, WORK / name
        if src.is_file() and not dst.exists():
            shutil.copy2(src, dst)
print("device:", DEVICE, "work:", WORK)
# Prefer Kaggle Secrets / env; leave unset to fall back to DINOv2
HF_TOKEN = os.getenv('HF_TOKEN') or os.getenv('HUGGING_FACE_HUB_TOKEN')


## Locate inputs

The local `_1000.csv` files are smoke-test slices and contain only
`same_figure`. Training a four-class fallback requires the full train CSV.


In [ ]:
def find_file(names, required=True):
    names = [names] if isinstance(names, str) else list(names)
    roots = [Path("/kaggle/input"), Path("data"), Path("."), Path("/kaggle/working")]
    hits = []
    for root in roots:
        if root.exists():
            for name in names:
                hits.extend(p for p in root.rglob(name) if p.is_file())
    hits = sorted(set(hits), key=lambda p: ("working" not in str(p), len(p.parts), str(p)))
    if not hits:
        if required:
            raise FileNotFoundError(names)
        return None
    print(names, "->", hits[0])
    return hits[0]

TRAIN_CSV = find_file(["train.csv", "train_1000.csv"])
TEST_CSV = find_file(["test.csv", "test_1000.csv"])
SAMPLE_CSV = find_file("sample_submission.csv")
MATCHER_SCRIPT = find_file("metadata_matching.py")
INDEX_CACHE = WORK / "astroclimb_metadata_index.pkl"
TRAIN_META = WORK / "train_metadata_matches.csv"
TEST_META = WORK / "test_metadata_matches.csv"

def csv_summary(path):
    header = pd.read_csv(path, nrows=0).columns
    total, counts = 0, Counter()
    for chunk in pd.read_csv(path, usecols=[c for c in ["id", *LABELS] if c in header], chunksize=1000):
        total += len(chunk)
        if set(LABELS).issubset(chunk.columns):
            counts.update(chunk[LABELS].idxmax(axis=1))
    return total, dict(counts)

print("train:", csv_summary(TRAIN_CSV))
print("test:", csv_summary(TEST_CSV))


## Exact HF matching

This invokes the repository matcher. It indexes captions first and then
streams only the image bytes needed by the pair CSV. The supplied archive
is used only for expensive embedding/features caches: archived match CSVs
and the metadata index are deliberately ignored because they can be stale.
A fresh index is reused between the train and test matcher passes.

The next cell enforces substantial test coverage. If it stops at 0%, the
test source records are absent from the attached/public metadata; a
deterministic metadata relationship cannot be recovered from that input.


In [ ]:
def run_matcher(pair_csv, output, rebuild_index=False):
    if output.exists() and not REFRESH_METADATA_MATCHES:
        print("reuse", output)
        return
    if not RUN_METADATA_MATCHER:
        raise FileNotFoundError(f"Missing {output}; enable RUN_METADATA_MATCHER")
    cmd = [sys.executable, str(MATCHER_SCRIPT), "--pairs", str(pair_csv),
           "--output", str(output), "--index-cache", str(INDEX_CACHE),
           "--image-batch-size", "32"]
    if METADATA_PATH is None:
        cmd.extend(["--hf-dataset", HF_DATASET])
    else:
        cmd.extend(["--metadata", str(METADATA_PATH)])
    if rebuild_index:
        cmd.append("--rebuild-index")
    print("running:", " ".join(cmd))
    subprocess.run(cmd, check=True)

run_matcher(TRAIN_CSV, TRAIN_META, rebuild_index=REBUILD_METADATA_INDEX)
run_matcher(TEST_CSV, TEST_META)


## Candidate-set graph reasoning

If an object has several exact metadata candidates, evaluate every candidate
pair. When all possible pairs imply the same graph relationship, that label is
safe even though neither object has a unique row match.


In [ ]:
def import_matcher(path):
    spec = importlib.util.spec_from_file_location("metadata_matching", path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[spec.name] = module
    spec.loader.exec_module(module)
    return module

MM = import_matcher(MATCHER_SCRIPT)

def load_metadata_index(path):
    import __main__
    for name in ("HuggingFaceMetadataIndex", "MetadataIndex", "Match"):
        if hasattr(MM, name):
            setattr(__main__, name, getattr(MM, name))
    with open(path, "rb") as handle:
        return pickle.load(handle)

def add_candidate_consensus(pair_csv, meta_csv, output):
    audit = pd.read_csv(meta_csv, keep_default_na=False)
    audit["graph_relationship"] = audit["inferred_relationship"]
    audit["resolution"] = np.where(audit.inferred_relationship.ne("unmatched"), "unique", "unresolved")
    wanted = set(audit.loc[audit.inferred_relationship.eq("unmatched"), "id"].astype(str))
    if not wanted or not RUN_CANDIDATE_CONSENSUS:
        audit.to_csv(output, index=False); return audit
    index = load_metadata_index(INDEX_CACHE)
    resolved = {}
    for chunk in pd.read_csv(pair_csv, usecols=["id", "obj_1", "obj_2"], chunksize=CSV_CHUNK, keep_default_na=False):
        for row in chunk.itertuples(index=False):
            if str(row.id) not in wanted:
                continue
            left, right = index.lookup(row.obj_1), index.lookup(row.obj_2)
            relationships = {
                MM.infer_relationship(index.records[i], index.records[j])
                for i in left.candidates for j in right.candidates
            }
            relationships.discard("unmatched")
            if len(relationships) == 1:
                resolved[str(row.id)] = relationships.pop()
    key = audit.id.astype(str)
    mask = key.isin(resolved)
    audit.loc[mask, "graph_relationship"] = key[mask].map(resolved)
    audit.loc[mask, "resolution"] = "candidate_consensus"
    audit.to_csv(output, index=False)
    print("candidate consensus recovered", int(mask.sum()), "rows")
    return audit

TRAIN_META_GRAPH = WORK / "train_metadata_graph.csv"
TEST_META_GRAPH = WORK / "test_metadata_graph.csv"
train_meta = add_candidate_consensus(TRAIN_CSV, TRAIN_META, TRAIN_META_GRAPH)
test_meta = add_candidate_consensus(TEST_CSV, TEST_META, TEST_META_GRAPH)
print(train_meta.resolution.value_counts())
print(test_meta.resolution.value_counts())
test_metadata_coverage = test_meta.graph_relationship.isin(LABELS).mean()
print(f"test metadata coverage: {test_metadata_coverage:.2%}")
if test_metadata_coverage < MIN_TEST_METADATA_COVERAGE:
    raise RuntimeError(
        f"Fresh metadata coverage is only {test_metadata_coverage:.2%}; required "
        f"{MIN_TEST_METADATA_COVERAGE:.0%}. The attached/public metadata does not "
        "contain the Kaggle test sources, so metadata-first inference is impossible. "
        "Attach a current, competition-permitted metadata snapshot containing the "
        "test figures, or lower MIN_TEST_METADATA_COVERAGE to explicitly allow "
        "model fallback."
    )


## Persistent representation cache and encoders


In [ ]:
def is_image(value):
    return isinstance(value, str) and value.lstrip().startswith(("iVBOR", "/9j/"))

def object_key(value):
    return hashlib.sha256(value.encode("utf-8")).hexdigest()

def decode_image(value):
    raw = base64.b64decode(value.strip(), validate=False)
    with Image.open(io.BytesIO(raw)) as image:
        image.thumbnail((2048, 2048))
        return image.convert("RGB").copy()

class VectorCache:
    def __init__(self, path):
        self.db = sqlite3.connect(path)
        self.db.execute("CREATE TABLE IF NOT EXISTS vectors(kind TEXT, key TEXT, dim INTEGER, value BLOB, PRIMARY KEY(kind,key))")
        self.db.execute("CREATE TABLE IF NOT EXISTS texts(kind TEXT, key TEXT, value TEXT, PRIMARY KEY(kind,key))")
    def get_vector(self, kind, key):
        row = self.db.execute("SELECT dim,value FROM vectors WHERE kind=? AND key=?", (kind,key)).fetchone()
        return None if row is None else np.frombuffer(row[1], dtype=np.float16, count=row[0]).astype(np.float32)
    def put_vector(self, kind, key, value):
        value = np.asarray(value, np.float16)
        self.db.execute("INSERT OR REPLACE INTO vectors VALUES (?,?,?,?)", (kind,key,len(value),value.tobytes()))
    def get_text(self, kind, key):
        row = self.db.execute("SELECT value FROM texts WHERE kind=? AND key=?", (kind,key)).fetchone()
        return None if row is None else row[0]
    def put_text(self, kind, key, value):
        self.db.execute("INSERT OR REPLACE INTO texts VALUES (?,?,?)", (kind,key,value))
    def commit(self): self.db.commit()

CACHE = VectorCache(WORK / "representations.sqlite")

specter_tokenizer = AutoTokenizer.from_pretrained(SPECTER_MODEL)
specter = AutoModel.from_pretrained(SPECTER_MODEL, dtype=DTYPE).eval().to(DEVICE)
siglip_processor = AutoProcessor.from_pretrained(SIGLIP_MODEL)
siglip = AutoModel.from_pretrained(SIGLIP_MODEL, dtype=DTYPE).eval().to(DEVICE)
try:
    dino_processor = AutoImageProcessor.from_pretrained(DINO3_MODEL, token=os.getenv("HF_TOKEN"))
    dino = AutoModel.from_pretrained(DINO3_MODEL, token=os.getenv("HF_TOKEN"), dtype=DTYPE).eval().to(DEVICE)
    ACTIVE_DINO = DINO3_MODEL
except Exception as exc:
    print("DINOv3 unavailable; using DINOv2:", type(exc).__name__)
    dino_processor = AutoImageProcessor.from_pretrained(DINO_FALLBACK)
    dino = AutoModel.from_pretrained(DINO_FALLBACK, dtype=DTYPE).eval().to(DEVICE)
    ACTIVE_DINO = DINO_FALLBACK

ocr_reader = None
if USE_OCR:
    try:
        import easyocr
        ocr_reader = easyocr.Reader(["en"], gpu=(DEVICE == "cuda"), verbose=False)
    except Exception as exc:
        print("OCR disabled:", exc)

def normalize(v):
    v = np.asarray(v, np.float32)
    return v / max(np.linalg.norm(v), 1e-8)

@torch.inference_mode()
def encode_specter(texts):
    batch = specter_tokenizer(texts, padding=True, truncation=True, max_length=512, return_tensors="pt")
    batch = {k:v.to(DEVICE) for k,v in batch.items()}
    return specter(**batch).last_hidden_state[:,0].float().cpu().numpy()

@torch.inference_mode()
def encode_siglip(values, image_flags):
    """Encode with SigLIP2; handle both tensor and BaseModelOutputWithPooling returns."""
    def _to_numpy(feats):
        if hasattr(feats, "pooler_output"):
            feats = feats.pooler_output
        elif hasattr(feats, "last_hidden_state"):
            feats = feats.last_hidden_state[:, 0]
        return feats.float().cpu().numpy()

    if all(image_flags):
        batch = siglip_processor(
            images=[decode_image(v) for v in values],
            padding="max_length",
            max_num_patches=256,
            return_tensors="pt",
        )
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        return _to_numpy(siglip.get_image_features(**batch))
    # text path — padding=max_length matches SigLIP2 training
    batch = siglip_processor(
        text=values,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    )
    batch = {k: v.to(DEVICE) for k, v in batch.items()}
    return _to_numpy(siglip.get_text_features(**batch))

@torch.inference_mode()
def encode_dino(values):
    batch = dino_processor(images=[decode_image(v) for v in values], return_tensors="pt")
    batch = {k: v.to(DEVICE) for k, v in batch.items()}
    out = dino(**batch)
    if hasattr(out, "pooler_output") and out.pooler_output is not None:
        feats = out.pooler_output
    else:
        feats = out.last_hidden_state[:, 0]
    return feats.float().cpu().numpy()

def cached_vectors(kind, values, eligible, encoder):
    result, missing = [None] * len(values), []
    for i,(value,ok) in enumerate(zip(values,eligible)):
        if not ok: continue
        key = object_key(value); cached = CACHE.get_vector(kind,key)
        if cached is None: missing.append(i)
        else: result[i] = cached
    for start in range(0,len(missing),MODEL_BATCH):
        idx = missing[start:start+MODEL_BATCH]
        encoded = encoder([values[i] for i in idx])
        for i,vector in zip(idx,encoded):
            vector=normalize(vector); result[i]=vector; CACHE.put_vector(kind,object_key(values[i]),vector)
        CACHE.commit()
    return result

def ocr_text(value):
    if ocr_reader is None or not is_image(value): return ""
    key=object_key(value); cached=CACHE.get_text("ocr",key)
    if cached is not None: return cached
    try: text=" ".join(ocr_reader.readtext(np.asarray(decode_image(value)),detail=0,paragraph=True))
    except Exception: text=""
    CACHE.put_text("ocr",key,text); CACHE.commit(); return text


## TF-IDF and handcrafted similarities


In [ ]:
TFIDF_PATH = WORK / "tfidf.joblib"
if TFIDF_PATH.exists():
    word_tfidf, char_tfidf = joblib.load(TFIDF_PATH)
else:
    captions = {}
    for chunk in pd.read_csv(TRAIN_CSV,usecols=["obj_1","obj_2"],chunksize=CSV_CHUNK,keep_default_na=False):
        for value in [*chunk.obj_1, *chunk.obj_2]:
            if not is_image(value): captions.setdefault(object_key(value),value)
    word_tfidf=TfidfVectorizer(ngram_range=(1,2),min_df=2,max_features=60000,strip_accents="unicode")
    char_tfidf=TfidfVectorizer(analyzer="char_wb",ngram_range=(3,5),min_df=2,max_features=60000)
    word_tfidf.fit(captions.values()); char_tfidf.fit(captions.values())
    joblib.dump((word_tfidf,char_tfidf),TFIDF_PATH); del captions; gc.collect()

def cosine_sparse(vectorizer,a,b):
    x,y=vectorizer.transform([a]),vectorizer.transform([b])
    return float(x.multiply(y).sum())

def pair_stats(a,b):
    if a is None or b is None: return [0.0,0.0,0.0,0.0]
    d=np.abs(a-b)
    return [float(a@b),float(d.mean()),float(np.linalg.norm(d)),float(d.max())]

def phash(value):
    if not is_image(value): return None
    image=decode_image(value).convert("L").resize((32,32),Image.Resampling.LANCZOS)
    coeff=dctn(np.asarray(image,np.float32),norm="ortho")[:8,:8]
    return int.from_bytes(np.packbits((coeff>np.median(coeff[1:])).ravel()).tobytes(),"big")

def token_jaccard(a,b):
    x,y=set(a.lower().split()),set(b.lower().split())
    return len(x&y)/max(len(x|y),1)

FEATURE_NAMES = [
    "specter_cos","specter_l1","specter_l2","specter_max",
    "siglip_cos","siglip_l1","siglip_l2","siglip_max",
    "dino_cos","dino_l1","dino_l2","dino_max",
    "word_tfidf","char_tfidf","phash_similarity","ocr_token_overlap","ocr_char_similarity",
    "text_text","text_image","image_image","length_1","length_2"
]


## Feature extraction


In [ ]:
def extract_features(csv_path, split, with_labels):
    cache_path=WORK/f"{split}_features.npz"
    if cache_path.exists():
        data=np.load(cache_path,allow_pickle=True)
        return data["ids"],data["X"].astype(np.float32),data["y"] if with_labels else None
    ids,features,targets=[],[],[]
    usecols=["id","obj_1","obj_2"]+ (LABELS if with_labels else [])
    for chunk_no,chunk in enumerate(pd.read_csv(csv_path,usecols=usecols,chunksize=CSV_CHUNK,keep_default_na=False)):
        left,right=chunk.obj_1.tolist(),chunk.obj_2.tolist(); values=left+right
        image_mask=[is_image(v) for v in values]; text_mask=[not x for x in image_mask]
        spec=cached_vectors("specter",values,text_mask,encode_specter)
        sig_text=cached_vectors("siglip",values,text_mask,lambda x:encode_siglip(x,[False]*len(x)))
        sig_image=cached_vectors("siglip",values,image_mask,lambda x:encode_siglip(x,[True]*len(x)))
        sig=[a if a is not None else b for a,b in zip(sig_text,sig_image)]
        din=cached_vectors("dino",values,image_mask,encode_dino)
        n=len(left)
        for i,(a,b) in enumerate(zip(left,right)):
            ai,bi=image_mask[i],image_mask[n+i]; tt=not ai and not bi; ii=ai and bi; mixed=ai^bi
            o1,o2=ocr_text(a),ocr_text(b)
            if mixed:
                caption=a if not ai else b; image_ocr=o2 if ai is False else o1
                ocr_tok=token_jaccard(caption,image_ocr); ocr_char=SequenceMatcher(None,caption.lower(),image_ocr.lower()).ratio()
            elif ii:
                ocr_tok=token_jaccard(o1,o2); ocr_char=SequenceMatcher(None,o1.lower(),o2.lower()).ratio()
            else: ocr_tok=ocr_char=0.0
            if tt:
                wt=cosine_sparse(word_tfidf,a,b); ct=cosine_sparse(char_tfidf,a,b)
            else: wt=ct=0.0
            if ii:
                pa,pb=phash(a),phash(b); ps=1.0-((pa^pb).bit_count()/64.0)
            else: ps=0.0
            row=pair_stats(spec[i],spec[n+i])+pair_stats(sig[i],sig[n+i])+pair_stats(din[i],din[n+i])
            row += [wt,ct,ps,ocr_tok,ocr_char,float(tt),float(mixed),float(ii),min(len(a),5000)/5000,min(len(b),5000)/5000]
            features.append(row); ids.append(chunk.id.iloc[i])
            if with_labels: targets.append(int(np.argmax(chunk.iloc[i][LABELS].astype(int).values)))
        if chunk_no%25==0: print(split,"rows",len(ids))
    X=np.asarray(features,np.float32); y=np.asarray(targets,np.int8)
    np.savez_compressed(cache_path,ids=np.asarray(ids),X=X.astype(np.float16),**({"y":y} if with_labels else {}))
    return np.asarray(ids),X,y if with_labels else None

train_ids,X,y=extract_features(TRAIN_CSV,"train",True)
if len(np.unique(y))<4:
    raise RuntimeError("The attached training CSV does not contain all four classes. Attach the full train.csv, not train_1000.csv.")
test_ids,X_test,_=extract_features(TEST_CSV,"test",False)
print(X.shape,dict(Counter(y)),FEATURE_NAMES)


## Unified Early-Fusion Neural Network (replaces 3 CatBoost specialists)

Instead of training three separate models and routing by modality, we train **one**
MLP on the full feature matrix. Modality indicators (`text_text`, `text_image`,
`image_image`) are part of the feature vector, so the network can condition its
behaviour on the available modalities while still sharing all parameters.

Architecture:
- Input: 22-dim pairwise features (cosine / L1 / L2 / max diffs of SPECTER, SigLIP, DINO
  + TF-IDF + pHash + OCR + modality flags + length)
- 3 residual MLP blocks with BatchNorm + GELU + Dropout
- Output: 4-class logits

Training uses StratifiedGroupKFold on DOI groups + class-balanced CrossEntropy.


In [ ]:
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, f1_score

# ----------------------------------------------------------------------
# Model definition
# ----------------------------------------------------------------------
class ResidualBlock(nn.Module):
    def __init__(self, dim, dropout=0.25):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
        )
        self.act = nn.GELU()
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.drop(self.act(x + self.net(x)))


class UnifiedFusionMLP(nn.Module):
    """Early-fusion MLP that sees all modalities at once."""
    def __init__(self, in_dim=22, hidden=256, n_blocks=3, n_classes=4, dropout=0.25):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.BatchNorm1d(hidden),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.blocks = nn.ModuleList([ResidualBlock(hidden, dropout) for _ in range(n_blocks)])
        self.head = nn.Linear(hidden, n_classes)

    def forward(self, x):
        h = self.stem(x)
        for blk in self.blocks:
            h = blk(h)
        return self.head(h)


# ----------------------------------------------------------------------
# Training helpers
# ----------------------------------------------------------------------
def train_one_fold(model, train_loader, val_loader, class_weights, epochs=40, lr=3e-3, device=DEVICE):
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))

    best_state, best_f1 = None, -1.0
    for epoch in range(epochs):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            opt.step()
        sched.step()

        # validation
        model.eval()
        preds, gts = [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                logits = model(xb.to(device))
                preds.append(logits.argmax(1).cpu())
                gts.append(yb)
        preds = torch.cat(preds).numpy()
        gts = torch.cat(gts).numpy()
        f1 = f1_score(gts, preds, average="macro")
        if f1 > best_f1:
            best_f1 = f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model, best_f1


def predict_proba(model, X, batch_size=1024, device=DEVICE):
    model.eval()
    loader = DataLoader(TensorDataset(torch.from_numpy(X)), batch_size=batch_size, shuffle=False)
    outs = []
    with torch.no_grad():
        for (xb,) in loader:
            logits = model(xb.to(device))
            outs.append(torch.softmax(logits, dim=1).cpu())
    return torch.cat(outs).numpy()


# ----------------------------------------------------------------------
# Prepare groups (DOI-based to reduce leakage)
# ----------------------------------------------------------------------
train_meta_by_id = train_meta.set_index("id").reindex(train_ids)
groups = (
    train_meta_by_id.obj_1_doi.fillna("").astype(str)
    + "|"
    + train_meta_by_id.obj_2_doi.fillna("").astype(str)
).values
groups = np.where(groups == "|", np.asarray([f"row-{i}" for i in range(len(train_ids))]), groups)

# ----------------------------------------------------------------------
# Cross-validation + final model
# ----------------------------------------------------------------------
N_FOLDS = 5
HIDDEN = 256
EPOCHS = 45
BATCH = 512
LR = 2.5e-3

oof = np.zeros((len(y), 4), np.float32)
fold_models = []

skf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y, groups)):
    print(f"\n=== Fold {fold+1}/{N_FOLDS} ===")
    X_tr, X_va = X[tr_idx], X[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]

    # class weights for this fold
    present = np.unique(y_tr)
    weights = compute_class_weight("balanced", classes=present, y=y_tr)
    weight_map = {int(c): float(w) for c, w in zip(present, weights)}
    class_w = torch.tensor([weight_map.get(i, 1.0) for i in range(4)], dtype=torch.float32)

    train_ds = TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr.astype(np.int64)))
    val_ds   = TensorDataset(torch.from_numpy(X_va), torch.from_numpy(y_va.astype(np.int64)))
    train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, drop_last=False)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH*2, shuffle=False)

    model = UnifiedFusionMLP(in_dim=X.shape[1], hidden=HIDDEN, n_blocks=3, dropout=0.3)
    model, fold_f1 = train_one_fold(model, train_loader, val_loader, class_w,
                                    epochs=EPOCHS, lr=LR, device=DEVICE)
    print(f"Fold {fold+1} best macro-F1: {fold_f1:.4f}")

    oof[va_idx] = predict_proba(model, X_va)
    fold_models.append(model)

print("\n=== Unified model OOF ===")
print("fallback OOF macro-F1", f1_score(y, oof.argmax(1), average="macro"))
print(classification_report(y, oof.argmax(1), target_names=LABELS, digits=4))

# Hybrid OOF (metadata override)
train_rule = train_meta_by_id.graph_relationship.fillna("unmatched").astype(str).values
hybrid_oof = oof.argmax(1).copy()
for class_id, label in enumerate(LABELS):
    hybrid_oof[train_rule == label] = class_id
print("hybrid OOF macro-F1", f1_score(y, hybrid_oof, average="macro"),
      "metadata coverage", float(np.isin(train_rule, LABELS).mean()))
print(classification_report(y, hybrid_oof, target_names=LABELS, digits=4))

# Train a final model on all data
print("\n=== Training final model on full data ===")
present = np.unique(y)
weights = compute_class_weight("balanced", classes=present, y=y)
class_w = torch.tensor([float(w) for w in weights], dtype=torch.float32)
full_ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(y.astype(np.int64)))
full_loader = DataLoader(full_ds, batch_size=BATCH, shuffle=True)
final_model = UnifiedFusionMLP(in_dim=X.shape[1], hidden=HIDDEN, n_blocks=3, dropout=0.3)
# quick final training (no early stopping needed for the last model)
final_model = final_model.to(DEVICE)
opt = torch.optim.AdamW(final_model.parameters(), lr=LR, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss(weight=class_w.to(DEVICE))
final_model.train()
for epoch in range(EPOCHS):
    for xb, yb in full_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        opt.zero_grad()
        loss = criterion(final_model(xb), yb)
        loss.backward()
        opt.step()
final_model.eval()

# Save
torch.save({
    "model_state": final_model.state_dict(),
    "in_dim": X.shape[1],
    "hidden": HIDDEN,
    "feature_names": FEATURE_NAMES,
}, WORK / "unified_fusion_mlp.pt")
print("Saved", WORK / "unified_fusion_mlp.pt")

## Hybrid inference and submission


In [ ]:
# ------------------------------------------------------------------
# Inference with the single unified model
# ------------------------------------------------------------------
fallback_proba = predict_proba(final_model, X_test)
fallback_label = np.asarray(LABELS, dtype=object)[fallback_proba.argmax(1)]

meta_test = test_meta.set_index("id").reindex(test_ids)
rule = meta_test.graph_relationship.fillna("unmatched").astype(str).values
resolved = np.isin(rule, LABELS)
final_label = np.where(resolved, rule, fallback_label)
print("metadata resolved", int(resolved.sum()), "/", len(resolved),
      "fallback", int((~resolved).sum()))

prediction = pd.DataFrame({"id": test_ids})
for label in LABELS:
    prediction[label] = (final_label == label).astype(np.int8)

sample = pd.read_csv(SAMPLE_CSV, usecols=["id", *LABELS])
submission = sample[["id"]].merge(prediction, on="id", how="left")
assert submission[LABELS].notna().all().all()
assert submission[LABELS].sum(axis=1).eq(1).all()

submission.to_csv(WORK / "submission_unified.csv", index=False)
pd.DataFrame({
    "id": test_ids,
    "metadata_relationship": rule,
    "resolution": meta_test.resolution.values,
    "fallback_relationship": fallback_label,
    "final_relationship": final_label,
    "fallback_confidence": fallback_proba.max(1),
}).to_csv(WORK / "prediction_audit_unified.csv", index=False)

print("saved", WORK / "submission_unified.csv")
display(submission.head())

## Recommended ablations & next steps

With the unified model in place, useful comparisons (same grouped folds):

- metadata only vs metadata + candidate consensus
- old 3× CatBoost specialists vs this unified MLP
- deeper / wider MLP, different dropout, residual vs plain
- adding raw embedding concatenation (higher dim, more GPU memory)
- LoRA fine-tuning of SigLIP2 / SPECTER2 on high-confidence metadata pairs
- cross-attention between the two objects’ embeddings instead of pairwise stats
- focal loss or multi-task auxiliary heads (same-DOI, citation-link)

Report macro-F1 by class, by modality, and by resolution type
(unique / candidate_consensus / unresolved).
